In [1]:
import os
os.environ["PYSPARK_PYTHON"] = r"C:\Users\Ben\AppData\Local\Programs\Python\Python310\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"C:\Users\Ben\AppData\Local\Programs\Python\Python310\python.exe"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import (
    col, to_date, to_timestamp, datediff, months_between,
    current_date, current_timestamp, year, month, dayofmonth,
    dayofweek, quarter, date_format, date_add, date_sub,
    round, when, sum, count, lit
)

spark = SparkSession.builder.appName("Week2_Day4").master("local[*]").getOrCreate()

# Employees with hire dates as STRINGS (messy, like real life)
emp_data = [
    (1, "Alice",   "Engineering", 95000, "2019-03-15"),
    (2, "Bob",     "Engineering", 90000, "2020-07-22"),
    (3, "Charlie", "Data",        85000, "2018-11-01"),
    (4, "Diana",   "Data",        80000, "2023-01-10"),
    (5, "Eve",     "HR",          70000, "2021-06-30"),
    (6, "Frank",   "HR",          65000, "2024-09-05"),
]

emp_schema = StructType([
    StructField("emp_id", IntegerType(), False),
    StructField("name", StringType(), False),
    StructField("department", StringType(), False),
    StructField("salary", IntegerType(), False),
    StructField("hire_date_str", StringType(), False),
])

df_emp = spark.createDataFrame(emp_data, emp_schema)

# Orders with timestamps
orders_data = [
    (101, "2024-01-15 09:30:00", 250.0),
    (102, "2024-01-20 14:15:00", 180.0),   # Saturday
    (103, "2024-02-03 11:00:00", 320.0),   # Saturday
    (104, "2024-02-14 16:45:00", 450.0),
    (105, "2024-03-08 08:20:00", 190.0),
    (106, "2024-03-17 13:10:00", 275.0),   # Sunday
    (107, "2024-04-01 10:00:00", 600.0),
    (108, "2024-04-21 15:30:00", 340.0),   # Sunday
    (109, "2024-05-06 09:45:00", 410.0),
    (110, "2024-06-15 12:00:00", 520.0),   # Saturday
]

orders_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("order_timestamp_str", StringType(), False),
    StructField("amount", DoubleType(), False),
])

df_orders = spark.createDataFrame(orders_data, orders_schema)

In [4]:
df_emp_clean = df_emp.withColumn(
    "hire_date",
    to_date(col("hire_date_str"), "yyyy-MM-dd")
)

df_emp_clean.select("name", "hire_date_str", "hire_date").show()

+-------+-------------+----------+
|   name|hire_date_str| hire_date|
+-------+-------------+----------+
|  Alice|   2019-03-15|2019-03-15|
|    Bob|   2020-07-22|2020-07-22|
|Charlie|   2018-11-01|2018-11-01|
|  Diana|   2023-01-10|2023-01-10|
|    Eve|   2021-06-30|2021-06-30|
|  Frank|   2024-09-05|2024-09-05|
+-------+-------------+----------+



In [8]:
# Different formats you'll encounter
messy_dates = [
    (1, "15/03/2024"),      # dd/MM/yyyy
    (2, "03-15-2024"),      # MM-dd-yyyy
    (3, "20240315"),        # yyyyMMdd
]

df_messy = spark.createDataFrame(messy_dates, ["id", "date_str"])

df_parsed =df_messy.withColumn(
    "parsed_v1",to_date(col("date_str"), "dd/MM/yyyy") 
    ).withColumn(
    "parsed_v2", to_date(col("date_str"), "MM-dd-yyyy")
    ).withColumn(
    "parsed_v3", to_date(col("date_str"), "yyyyMMdd")
    )

df_parsed.show()

+---+----------+----------+----------+----------+
| id|  date_str| parsed_v1| parsed_v2| parsed_v3|
+---+----------+----------+----------+----------+
|  1|15/03/2024|2024-03-15|      NULL|      NULL|
|  2|03-15-2024|      NULL|2024-03-15|      NULL|
|  3|  20240315|      NULL|      NULL|2024-03-15|
+---+----------+----------+----------+----------+



In [9]:
df_orders_clean = df_orders.withColumn(
    "order_timestamp",
    to_timestamp(col("order_timestamp_str"), "yyyy-MM-dd HH:mm:ss")
).withColumn(
    "order_date",
    to_date(col("order_timestamp_str"), "yyyy-MM-dd HH:mm:ss")
)

df_orders_clean.show()

+--------+-------------------+------+-------------------+----------+
|order_id|order_timestamp_str|amount|    order_timestamp|order_date|
+--------+-------------------+------+-------------------+----------+
|     101|2024-01-15 09:30:00| 250.0|2024-01-15 09:30:00|2024-01-15|
|     102|2024-01-20 14:15:00| 180.0|2024-01-20 14:15:00|2024-01-20|
|     103|2024-02-03 11:00:00| 320.0|2024-02-03 11:00:00|2024-02-03|
|     104|2024-02-14 16:45:00| 450.0|2024-02-14 16:45:00|2024-02-14|
|     105|2024-03-08 08:20:00| 190.0|2024-03-08 08:20:00|2024-03-08|
|     106|2024-03-17 13:10:00| 275.0|2024-03-17 13:10:00|2024-03-17|
|     107|2024-04-01 10:00:00| 600.0|2024-04-01 10:00:00|2024-04-01|
|     108|2024-04-21 15:30:00| 340.0|2024-04-21 15:30:00|2024-04-21|
|     109|2024-05-06 09:45:00| 410.0|2024-05-06 09:45:00|2024-05-06|
|     110|2024-06-15 12:00:00| 520.0|2024-06-15 12:00:00|2024-06-15|
+--------+-------------------+------+-------------------+----------+

